# LeetCode 659 — Split Array into Consecutive Subsequences bằng Python

Link đề: https://leetcode.com/problems/split-array-into-consecutive-subsequences/ · Độ khó: **Medium** · Chủ đề: Greedy, Array, Hash Table, Heap

Skeleton LeetCode:

```python
class Solution:
    def isPossible(self, nums: List[int]) -> bool:
        ...
```

Ghi chú: mảng `nums` đã **sắp xếp không giảm**, có thể có số trùng nhau. Ta phải chia **mọi** phần tử vào các "dãy con liên tiếp tăng" (mỗi số hơn số trước đúng 1) sao cho **dãy nào cũng dài ít nhất 3**. Đây là bài **tham lam (greedy)**: duyệt từ nhỏ đến lớn, mỗi số hoặc **nối vào một dãy đang chờ nó**, hoặc **mở dãy mới**. Notebook đi từ vét cạn (thử mọi cách gán), qua tham lam với hai bảng đếm, tham lam bằng heap độ dài, và tham lam nén trạng thái 3 số.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chia **toàn bộ** mảng đã sắp xếp thành các dãy con mà mỗi dãy gồm các số **liên tiếp tăng dần (hơn kém 1)** và **dài ≥ 3**; trả `True` nếu chia được.
2. Nắm các công cụ Python: `collections.Counter` / `defaultdict`, `heapq` (min-heap), `itertools.groupby`, `all(...)`, và ý niệm **dãy con** (subsequence) khác với **đoạn con** (subarray).
3. Đi từ **vét cạn mọi cách gán số vào dãy** → **tham lam hai bảng đếm `cnt` + `tail`** → **tham lam bằng heap độ dài (nối vào dãy ngắn nhất)** → **tham lam nén trạng thái (số dãy độ dài 1, 2, ≥3)**.
4. Hiểu mấu chốt: với mỗi số `v`, **nếu có dãy đang kết thúc ở `v - 1` thì nối vào luôn** (không bao giờ tệ hơn mở dãy mới); nếu không thì **bắt buộc** mở dãy mới `v, v+1, v+2`, thiếu `v+1` hoặc `v+2` là không thể.
5. Tránh các lỗi kinh điển: chỉ kiểm tra "các giá trị phân biệt liên tiếp ≥ 3" (bỏ qua số trùng), nối vào dãy **dài nhất** thay vì **ngắn nhất**, quên kiểm tra độ dài ≥ 3 ở cuối, và quên bỏ qua số đã được dùng để mở dãy (`cnt[v] == 0`).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` **sắp xếp không giảm**. Xác định xem có thể chia `nums` thành **một hay nhiều dãy con** sao cho **cả hai** điều kiện sau đúng:

- Mỗi dãy con là một **dãy tăng liên tiếp** (mỗi số bằng số liền trước cộng đúng `1`).
- Mọi dãy con đều có độ dài **từ 3 trở lên**.

Trả về `True` nếu chia được, ngược lại `False`.

Dãy con (subsequence) là dãy nhận được bằng cách xóa một số phần tử (có thể không xóa) mà **giữ nguyên thứ tự** các phần tử còn lại. Ví dụ `[1,3,5]` là dãy con của `[1,2,3,4,5]`, còn `[1,3,2]` thì không.

**Ví dụ 1**

- Input: `nums = [1,2,3,3,4,5]`
- Output: `True`
- Giải thích: chia thành `1,2,3` và `3,4,5`.

**Ví dụ 2**

- Input: `nums = [1,2,3,3,4,4,5,5]`
- Output: `True`
- Giải thích: chia thành `1,2,3,4,5` và `3,4,5`.

**Ví dụ 3**

- Input: `nums = [1,2,3,4,4,5]`
- Output: `False`
- Giải thích: không thể chia được (số `4` thứ hai và `5` không đủ để lập dãy dài 3).

**Ràng buộc:**

- `1 <= nums.length <= 10^4`
- `-1000 <= nums[i] <= 1000`
- `nums` sắp xếp **không giảm** (có thể có số trùng nhau)

Chú ý: **mọi** phần tử phải được dùng (không được bỏ số nào), và mỗi phần tử thuộc đúng một dãy.

In [1]:
from typing import List
from collections import Counter, defaultdict
from functools import lru_cache
from itertools import groupby, combinations_with_replacement
import heapq
import random
import time

examples = [
    ([1, 2, 3, 3, 4, 5], True),
    ([1, 2, 3, 3, 4, 4, 5, 5], True),
    ([1, 2, 3, 4, 4, 5], False),
]
for nums, expected in examples:
    print("nums =", nums, "| đếm:", dict(Counter(nums)), "| kỳ vọng:", expected)

nums = [1, 2, 3, 3, 4, 5] | đếm: {1: 1, 2: 1, 3: 2, 4: 1, 5: 1} | kỳ vọng: True
nums = [1, 2, 3, 3, 4, 4, 5, 5] | đếm: {1: 1, 2: 1, 3: 2, 4: 2, 5: 2} | kỳ vọng: True
nums = [1, 2, 3, 4, 4, 5] | đếm: {1: 1, 2: 1, 3: 1, 4: 2, 5: 1} | kỳ vọng: False


## 3. Kiến thức nền

### 3.1. Dãy con liên tiếp và "kết thúc ở `v`"

Một dãy liên tiếp như `[3,4,5]` có **số kết thúc** (đuôi) là `5`. Số `6` có thể **nối** vào dãy này (làm đuôi mới thành `6`); số `7` hay `5` thì không. Vì thế ta chỉ cần biết, tại mỗi thời điểm, **có bao nhiêu dãy đang kết thúc ở giá trị nào**. Dãy `[1,2,3,3,4,5]` chia thành `[1,2,3]` (đuôi `3`) và `[3,4,5]` (đuôi `5`).

In [2]:
chains = [[1, 2, 3], [3, 4, 5]]
tails = Counter(c[-1] for c in chains)
print("đuôi các dãy:", dict(tails))
print("có dãy nào đang chờ số 4 không? (cần dãy kết thúc ở 3):", tails[3] > 0)
print("có dãy nào đang chờ số 7 không? (cần dãy kết thúc ở 6):", tails[6] > 0)

đuôi các dãy: {3: 1, 5: 1}
có dãy nào đang chờ số 4 không? (cần dãy kết thúc ở 3): True
có dãy nào đang chờ số 7 không? (cần dãy kết thúc ở 6): False


### 3.2. `collections.Counter` và truy cập khóa vắng mặt

`Counter(nums)` đếm số lần xuất hiện của mỗi giá trị. Truy cập khóa **không có** trả `0` và **không** thêm khóa đó vào (khác `defaultdict`). Tiện để viết `cnt[v + 1] > 0` mà không lo lỗi `KeyError`. Các toán tử `cnt[v] -= 1` cũng dùng được trực tiếp.

In [3]:
cnt = Counter([1, 2, 2, 3])
print(cnt, cnt[2], cnt[99])
print(99 in cnt)              # False: truy cập cnt[99] không tạo khóa
cnt[2] -= 1
print(cnt)

Counter({2: 2, 1: 1, 3: 1}) 2 0
False
Counter({1: 1, 2: 1, 3: 1})


### 3.3. `defaultdict(list)` và heap độ dài

`defaultdict(list)` là từ điển tự tạo `[]` khi truy cập khóa mới. Cách 3 dùng `d[v]` là một **min-heap** (`heapq`) chứa độ dài của các dãy đang kết thúc ở `v`. `heapq.heappush(h, x)` thêm, `heapq.heappop(h)` lấy phần tử **nhỏ nhất** ra, `h[0]` xem phần tử nhỏ nhất.

In [4]:
d = defaultdict(list)
heapq.heappush(d[3], 5)
heapq.heappush(d[3], 3)
heapq.heappush(d[3], 4)
print(d[3][0], "là độ dài nhỏ nhất")
print(heapq.heappop(d[3]), "vừa lấy ra; còn", sorted(d[3]))
print(d[10])        # khóa mới -> danh sách rỗng

3 là độ dài nhỏ nhất
3 vừa lấy ra; còn [4, 5]
[]


### 3.4. `itertools.groupby` — nhóm các số bằng nhau liên tiếp

Vì `nums` đã **sắp xếp**, các số bằng nhau nằm liền kề; `groupby` chia thành từng nhóm (giá trị, số lần xuất hiện) chỉ bằng một lượt duyệt, không cần `Counter`. Cách 4 dùng để duyệt các **giá trị phân biệt** theo thứ tự tăng dần.

In [5]:
nums = [1, 2, 3, 3, 4, 4, 5, 5]
runs = [(v, len(list(g))) for v, g in groupby(nums)]
print(runs)

[(1, 1), (2, 1), (3, 2), (4, 2), (5, 2)]


### 3.5. `all(...)` và `any(...)`

`all(điều_kiện for ...)` đúng khi **mọi** điều kiện đúng (đúng với dãy rỗng); `any(...)` đúng khi **có ít nhất một**. Dùng để kiểm tra "mọi dãy đều dài ≥ 3".

In [6]:
lengths = [3, 5, 4]
print(all(l >= 3 for l in lengths), any(l < 3 for l in lengths))
lengths = [3, 2]
print(all(l >= 3 for l in lengths), any(l < 3 for l in lengths))
print(all([]), any([]))

True False
False True
True False


## 4. Ý tưởng nền tảng / trực giác

Hãy tự chia tay các ví dụ.

**`[1,2,3,3,4,5]` (True):** Duyệt từ trái. `1` chưa có dãy nào để nối → mở dãy mới; để dãy hợp lệ phải có `2,3` đi theo: `1,2,3`. Số `3` thứ hai không nối vào dãy nào đang chờ `3` (dãy `1,2,3` đã có đuôi `3`, cần số `4` để nối), nên mở dãy mới `3,4,5`. Xong, mọi dãy dài 3.

**`[1,2,3,4,4,5]` (False):** `1,2,3` mở dãy; `4` đầu tiên nối vào dãy `1,2,3` → `1,2,3,4`. `4` thứ hai không có dãy nào kết thúc ở `3` nữa → phải mở dãy mới `4,5,6`, nhưng **không có `6`** → thất bại. Thử cách khác: `4` đầu tiên *không* nối mà mở dãy mới? Cũng cần `5,6`, vẫn thiếu `6`. Không có cách nào → `False`.

Hai nhận xét dẫn đường:

- Các số xử lý theo **thứ tự tăng dần**, nên khi gặp `v`, mọi số nhỏ hơn đã được xếp xong; chỉ cần biết **có dãy nào đang chờ `v`** (đuôi `v - 1`).
- Khi gặp `v` mà **không** có dãy nào chờ, `v` buộc phải **mở dãy mới**. Dãy mới cần ≥ 3 phần tử liên tiếp bắt đầu từ `v`, nên cần có `v + 1` và `v + 2`.

Câu hỏi: nếu **cả hai** lựa chọn đều khả thi (nối vào dãy đang chờ, hoặc mở dãy mới), nên chọn cái nào? Mục 6 chứng minh: **luôn nối vào**.

## 5. Cách 1 — Brute Force: thử mọi cách gán (DFS + nhớ)

**Ý tưởng.** Duyệt từng phần tử theo thứ tự. Trạng thái là **danh sách các dãy đang có**, mỗi dãy biểu diễn bằng `(đuôi, độ dài)`. Với phần tử `v` có hai loại lựa chọn: (a) nối vào một dãy có đuôi `v - 1` (mỗi loại dãy `(đuôi, độ dài)` khác nhau là một lựa chọn khác nhau); (b) mở dãy mới `(v, 1)`. Sau khi xử lý hết, hợp lệ nếu **mọi dãy có độ dài ≥ 3**. Dùng `lru_cache` để không giải lại cùng một trạng thái.

Hàm có tham số `minlen` (độ dài tối thiểu, mặc định 3) và `maxlen` (độ dài tối đa, mặc định không giới hạn) để dùng lại cho các biến thể ở mục 15. Cách 1 cũng là **oracle** kiểm chứng.

In [7]:
def split_brute(nums, minlen=3, maxlen=None):
    nums = tuple(nums)

    @lru_cache(maxsize=None)
    def go(i, chains):
        if i == len(nums):
            return all(length >= minlen for _, length in chains)
        v = nums[i]
        tried = set()
        for k, (end, length) in enumerate(chains):
            if end == v - 1 and (end, length) not in tried and (maxlen is None or length < maxlen):
                tried.add((end, length))
                new_chains = list(chains)
                new_chains[k] = (v, length + 1)               # nối v vào dãy này
                if go(i + 1, tuple(sorted(new_chains))):
                    return True
        return go(i + 1, tuple(sorted(chains + ((v, 1),))))   # mở dãy mới

    return go(0, ())


for nums, expected in examples:
    got = split_brute(nums)
    print(nums, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected

[1, 2, 3, 3, 4, 5] -> True (kỳ vọng True )
[1, 2, 3, 3, 4, 4, 5, 5] -> True (kỳ vọng True )
[1, 2, 3, 4, 4, 5] -> False (kỳ vọng False )


**Độ phức tạp:** Time **mũ** trong trường hợp xấu (số trạng thái các dãy đang mở có thể rất lớn); chỉ dùng được cho mảng nhỏ (khoảng ≤ 15 phần tử). Space bằng số trạng thái được nhớ.

**Nút thắt:** `n` tới `10^4` thì không thể thử mọi cách gán. Cần một **quy tắc chọn duy nhất** cho mỗi phần tử mà luôn an toàn: đó là bước đột phá ở mục 6.

## 6. Ý tưởng thuật toán / bước đột phá

Duyệt các số theo thứ tự tăng dần. Với số `v` đang xét, xảy ra ba tình huống:

1. **Có dãy đang kết thúc ở `v - 1`** → **nối `v` vào dãy đó** (luôn làm vậy).
2. **Không có**, nhưng còn đủ `v + 1` và `v + 2` chưa dùng → **mở dãy mới `v, v+1, v+2`** (đánh dấu đuôi `v + 2`).
3. Còn lại → **thất bại** (`False`).

**Vì sao "nối vào" không bao giờ tệ hơn "mở dãy mới"?** Giả sử lời giải hợp lệ nào đó mở dãy mới `C` bắt đầu bằng `v` trong khi có dãy `D` kết thúc ở `v - 1`. Nếu `D` kết thúc tại `v - 1`, ta **nối `D` và `C` thành một dãy dài hơn**: vẫn liên tiếp, vẫn dài ≥ 3, số dãy giảm đi: hợp lệ. (Nếu `D` về sau được một số `v` khác nối tiếp, hai số `v` bằng nhau nên **hoán đổi vai trò** cho nhau không đổi gì.) Vậy luôn tồn tại lời giải trong đó `v` nối vào `D`.

**Vì sao khi không nối được thì buộc mở dãy `v, v+1, v+2`?** Dãy chứa `v` mà `v` không có số đứng trước phải **bắt đầu** ở `v`, và cần ít nhất ba phần tử liên tiếp: `v, v+1, v+2`. Nếu thiếu một trong hai số sau thì không thể. Còn lấy đúng ba số rồi để các số `v+3, v+4, ...` (nếu có) tự **nối vào đuôi `v+2`** theo quy tắc 1 là đủ.

**Cấu trúc dữ liệu.** Hai bảng đếm đủ dùng:

- `cnt[v]`: còn bao nhiêu số `v` **chưa được dùng**;
- `tail[v]`: có bao nhiêu dãy đang **kết thúc ở `v`** (đều đã dài ≥ 3, vì dãy luôn được tạo với 3 phần tử).

Vì mọi dãy sinh ra từ quy tắc 2 đều dài ≥ 3, **không cần theo dõi độ dài**; chỉ cần đếm đuôi.

## 7. Các cách tiếp theo

Phần này có thêm 3 cách nữa (tổng 4 cách): tham lam hai bảng đếm (cách nên nộp), tham lam bằng heap độ dài (nối vào dãy **ngắn nhất**), và tham lam nén trạng thái 3 số.

### 7.1. Cách 2 — Tham lam với hai bảng đếm `cnt` và `tail`

**Giải thích từng đoạn.**

1. `cnt = Counter(nums)`: đếm số lần xuất hiện, `tail = Counter()` ban đầu rỗng.
2. Duyệt từng `v` trong `nums` (theo thứ tự). Nếu `cnt[v] == 0` thì số này đã được dùng (đã làm thành viên của một dãy mở ở bước trước) → bỏ qua.
3. Nếu `tail[v - 1] > 0`: nối vào dãy chờ: `tail[v-1] -= 1`, `tail[v] += 1`, `cnt[v] -= 1`.
4. Ngược lại nếu `cnt[v + 1] > 0` và `cnt[v + 2] > 0`: mở dãy `v, v+1, v+2`: trừ cả ba `cnt`, `tail[v + 2] += 1`.
5. Ngược lại: trả `False`. Duyệt xong trả `True`.

In [8]:
def split_two_maps(nums):
    cnt = Counter(nums)
    tail = Counter()
    for v in nums:
        if cnt[v] == 0:
            continue                                   # v đã được dùng khi mở một dãy trước đó
        if tail[v - 1] > 0:                            # có dãy đang chờ v
            tail[v - 1] -= 1
            tail[v] += 1
            cnt[v] -= 1
        elif cnt[v + 1] > 0 and cnt[v + 2] > 0:        # mở dãy mới v, v+1, v+2
            cnt[v] -= 1
            cnt[v + 1] -= 1
            cnt[v + 2] -= 1
            tail[v + 2] += 1
        else:
            return False
    return True


for nums, expected in examples:
    got = split_two_maps(nums)
    print(nums, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected

[1, 2, 3, 3, 4, 5] -> True (kỳ vọng True )
[1, 2, 3, 3, 4, 4, 5, 5] -> True (kỳ vọng True )
[1, 2, 3, 4, 4, 5] -> False (kỳ vọng False )


In [9]:
class Solution:
    def isPossible(self, nums: List[int]) -> bool:
        cnt = Counter(nums)
        tail = Counter()
        for v in nums:
            if cnt[v] == 0:
                continue
            if tail[v - 1] > 0:
                tail[v - 1] -= 1
                tail[v] += 1
                cnt[v] -= 1
            elif cnt[v + 1] > 0 and cnt[v + 2] > 0:
                cnt[v] -= 1
                cnt[v + 1] -= 1
                cnt[v + 2] -= 1
                tail[v + 2] += 1
            else:
                return False
        return True

print([Solution().isPossible(n) for n, _ in examples])

[True, True, False]


**Độ phức tạp:** Time `O(n)` (mỗi phần tử xử lý một lần, thao tác bảng đếm `O(1)`). Space `O(m)` với `m` là số giá trị phân biệt (`m ≤ 2001` theo ràng buộc, `m ≤ n` nói chung).

### 7.2. Cách 3 — Heap độ dài: nối vào dãy **ngắn nhất**

**Giải thích từng đoạn.** Thay vì mở ngay dãy ba phần tử, cách này cho phép **mở dãy độ dài 1** và theo dõi **độ dài** từng dãy. `d[v]` là min-heap các độ dài của dãy đang kết thúc ở `v`. Với mỗi `v` (duyệt theo thứ tự): nếu `d[v - 1]` không rỗng thì lấy **độ dài nhỏ nhất** `l`, đẩy `l + 1` vào `d[v]` (nối vào dãy **ngắn nhất** để cứu các dãy còn quá ngắn); nếu rỗng thì đẩy `1` vào `d[v]` (mở dãy mới độ dài 1). Cuối cùng, mọi dãy phải dài ≥ 3: mỗi heap khác rỗng phải có phần tử nhỏ nhất ≥ 3.

In [10]:
def split_heap(nums):
    d = defaultdict(list)
    for v in nums:
        h = d[v - 1]
        if h:
            heapq.heappush(d[v], heapq.heappop(h) + 1)   # nối vào dãy ngắn nhất kết thúc ở v-1
        else:
            heapq.heappush(d[v], 1)                      # mở dãy mới độ dài 1
    return all(not h or h[0] >= 3 for h in d.values())


for nums, expected in examples:
    got = split_heap(nums)
    print(nums, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected

[1, 2, 3, 3, 4, 5] -> True (kỳ vọng True )
[1, 2, 3, 3, 4, 4, 5, 5] -> True (kỳ vọng True )
[1, 2, 3, 4, 4, 5] -> False (kỳ vọng False )


In [11]:
class Solution:
    def isPossible(self, nums: List[int]) -> bool:
        d = defaultdict(list)
        for v in nums:
            h = d[v - 1]
            if h:
                heapq.heappush(d[v], heapq.heappop(h) + 1)
            else:
                heapq.heappush(d[v], 1)
        return all(not h or h[0] >= 3 for h in d.values())

print([Solution().isPossible(n) for n, _ in examples])

[True, True, False]


**Độ phức tạp:** Time `O(n log n)` (mỗi phần tử một lần `heappush`/`heappop` trên heap tối đa `n` phần tử). Space `O(n)`. Chậm hơn cách 2 một hệ số `log` nhưng **không cần nhìn trước** `v+1`, `v+2`, chỉ ra quyết định tại chỗ rồi kiểm tra lại cuối cùng.

**Vì sao ngắn nhất?** Dãy ngắn là dãy "nguy hiểm" (chưa đạt 3). Nối vào dãy ngắn nhất đưa nó gần mục tiêu 3 hơn; nếu nối vào dãy dài thì dãy ngắn bị bỏ rơi. Mục 12.2 có phản ví dụ khi chọn dãy **dài nhất**.

### 7.3. Cách 4 — Nén trạng thái: số dãy độ dài 1, 2, ≥ 3

**Giải thích từng đoạn.** Heap thực ra chỉ cần phân biệt ba loại độ dài: `1`, `2`, `≥ 3` (từ 3 trở lên đều "an toàn" và không cần biết cụ thể). Xử lý **theo từng giá trị phân biệt** `v` với số lần xuất hiện `c`: gọi `p1, p2, p3` là số dãy đang kết thúc ở `v - 1` có độ dài `1`, `2`, `≥ 3`.

- Các dãy độ dài 1 và 2 **bắt buộc** phải được nối thêm `v` (nếu không, chúng kẹt ở độ dài < 3). Vậy cần `c ≥ p1 + p2`, nếu không → `False`.
- Còn lại `rem = c - p1 - p2` số `v`: ưu tiên nối vào dãy dài `≥ 3` (số dãy nối được `min(p3, rem)`), phần dư mới mở dãy mới độ dài 1.
- Trạng thái mới tại `v`: `p1' = max(0, rem - p3)`, `p2' = p1`, `p3' = p2 + min(p3, rem)`.
- Nếu `v` **không** liền sau giá trị trước (có khoảng trống), mọi dãy ở trước phải đã dài ≥ 3 (`p1 = p2 = 0`) rồi trạng thái được đặt lại `0, 0, 0`.
- Cuối cùng cần `p1 = p2 = 0`.

In [12]:
def split_three_states(nums):
    p1 = p2 = p3 = 0                    # số dãy kết thúc ở giá trị trước, độ dài 1 / 2 / >=3
    prev = None
    for v, group in groupby(nums):      # nums đã sắp xếp: các giá trị phân biệt tăng dần
        c = sum(1 for _ in group)
        if prev is not None and v != prev + 1:
            if p1 or p2:
                return False            # có dãy ngắn kẹt lại, không còn số nào để nối
            p1 = p2 = p3 = 0
        if c < p1 + p2:
            return False                # không đủ số v để cứu các dãy độ dài 1 và 2
        rem = c - p1 - p2
        p1, p2, p3 = max(0, rem - p3), p1, p2 + min(p3, rem)
        prev = v
    return p1 == 0 and p2 == 0


for nums, expected in examples:
    got = split_three_states(nums)
    print(nums, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected

[1, 2, 3, 3, 4, 5] -> True (kỳ vọng True )
[1, 2, 3, 3, 4, 4, 5, 5] -> True (kỳ vọng True )
[1, 2, 3, 4, 4, 5] -> False (kỳ vọng False )


In [13]:
class Solution:
    def isPossible(self, nums: List[int]) -> bool:
        p1 = p2 = p3 = 0
        prev = None
        for v, group in groupby(nums):
            c = sum(1 for _ in group)
            if prev is not None and v != prev + 1:
                if p1 or p2:
                    return False
                p1 = p2 = p3 = 0
            if c < p1 + p2:
                return False
            rem = c - p1 - p2
            p1, p2, p3 = max(0, rem - p3), p1, p2 + min(p3, rem)
            prev = v
        return p1 == 0 and p2 == 0

print([Solution().isPossible(n) for n, _ in examples])

[True, True, False]


**Độ phức tạp:** Time `O(n)` (một lượt duyệt, `groupby` không cần sắp xếp lại vì đã có thứ tự). Space `O(1)` (ba số). Đây là cách tiết kiệm nhất; nhược điểm là khó nghĩ hơn.

## 8. TRACE từng bước

Trace cách 2 (hai bảng đếm) với `nums = [1,2,3,3,4,4,5,5]`, kỳ vọng `True`. Ban đầu `cnt = {1:1, 2:1, 3:2, 4:2, 5:2}`, `tail` rỗng.

| bước | `v` | `cnt[v]` trước | hành động | `cnt` sau | `tail` sau |
|---|---|---|---|---|---|
| 1 | 1 | 1 | `tail[0] = 0`; có `2` và `3` → mở dãy `1,2,3` | `{1:0, 2:0, 3:1, 4:2, 5:2}` | `{3:1}` |
| 2 | 2 | 0 | bỏ qua (đã dùng) | không đổi | `{3:1}` |
| 3 | 3 | 1 | `tail[2] = 0`; có `4`, `5` → mở dãy `3,4,5` | `{3:0, 4:1, 5:1}` | `{3:1, 5:1}` |
| 4 | 3 | 0 | bỏ qua | không đổi | `{3:1, 5:1}` |
| 5 | 4 | 1 | `tail[3] = 1` → nối: dãy `1,2,3,4` | `{4:0}` | `{4:1, 5:1}` |
| 6 | 4 | 0 | bỏ qua | không đổi | `{4:1, 5:1}` |
| 7 | 5 | 1 | `tail[4] = 1` → nối: dãy `1,2,3,4,5` | `{5:0}` | `{5:2}` |
| 8 | 5 | 0 | bỏ qua | không đổi | `{5:2}` |

Duyệt hết không thất bại → `True` (hai dãy `1,2,3,4,5` và `3,4,5`).

In [14]:
def trace_two_maps(nums):
    cnt = Counter(nums)
    tail = Counter()
    print("nums =", nums)
    print("cnt ban đầu =", dict(sorted(cnt.items())))
    for step, v in enumerate(nums, 1):
        if cnt[v] == 0:
            print(f"bước {step}: v={v} -> cnt[v]=0, đã dùng, bỏ qua")
            continue
        if tail[v - 1] > 0:
            tail[v - 1] -= 1
            tail[v] += 1
            cnt[v] -= 1
            action = f"nối vào dãy kết thúc ở {v - 1}"
        elif cnt[v + 1] > 0 and cnt[v + 2] > 0:
            cnt[v] -= 1
            cnt[v + 1] -= 1
            cnt[v + 2] -= 1
            tail[v + 2] += 1
            action = f"mở dãy {v},{v + 1},{v + 2}"
        else:
            print(f"bước {step}: v={v} -> không nối được và không đủ {v + 1}, {v + 2} -> False")
            return False
        live_cnt = {k: c for k, c in sorted(cnt.items()) if c}
        live_tail = {k: c for k, c in sorted(tail.items()) if c}
        print(f"bước {step}: v={v} -> {action}; cnt={live_cnt}; tail={live_tail}")
    print("duyệt hết -> True")
    return True

assert trace_two_maps([1, 2, 3, 3, 4, 4, 5, 5]) is True

nums = [1, 2, 3, 3, 4, 4, 5, 5]
cnt ban đầu = {1: 1, 2: 1, 3: 2, 4: 2, 5: 2}
bước 1: v=1 -> mở dãy 1,2,3; cnt={3: 1, 4: 2, 5: 2}; tail={3: 1}
bước 2: v=2 -> cnt[v]=0, đã dùng, bỏ qua
bước 3: v=3 -> mở dãy 3,4,5; cnt={4: 1, 5: 1}; tail={3: 1, 5: 1}
bước 4: v=3 -> cnt[v]=0, đã dùng, bỏ qua
bước 5: v=4 -> nối vào dãy kết thúc ở 3; cnt={5: 1}; tail={4: 1, 5: 1}
bước 6: v=4 -> cnt[v]=0, đã dùng, bỏ qua
bước 7: v=5 -> nối vào dãy kết thúc ở 4; cnt={}; tail={5: 2}
bước 8: v=5 -> cnt[v]=0, đã dùng, bỏ qua
duyệt hết -> True


Trace ví dụ 3 `nums = [1,2,3,4,4,5]` (kỳ vọng `False`):

| bước | `v` | hành động | `tail` sau |
|---|---|---|---|
| 1 | 1 | mở dãy `1,2,3` | `{3:1}` |
| 2, 3 | 2, 3 | đã dùng, bỏ qua | `{3:1}` |
| 4 | 4 | nối vào dãy kết thúc ở 3 | `{4:1}` |
| 5 | 4 | `tail[3] = 0`; `cnt[5] = 1` nhưng `cnt[6] = 0` → **False** | |

In [15]:
assert trace_two_maps([1, 2, 3, 4, 4, 5]) is False
print()
assert trace_two_maps([1, 2, 3, 3, 4, 5]) is True
print()
# ca có khoảng trống: hai cụm tách rời đều hợp lệ
assert trace_two_maps([1, 2, 3, 7, 8, 9, 9, 10, 11]) is True

nums = [1, 2, 3, 4, 4, 5]
cnt ban đầu = {1: 1, 2: 1, 3: 1, 4: 2, 5: 1}
bước 1: v=1 -> mở dãy 1,2,3; cnt={4: 2, 5: 1}; tail={3: 1}
bước 2: v=2 -> cnt[v]=0, đã dùng, bỏ qua
bước 3: v=3 -> cnt[v]=0, đã dùng, bỏ qua
bước 4: v=4 -> nối vào dãy kết thúc ở 3; cnt={4: 1, 5: 1}; tail={4: 1}
bước 5: v=4 -> không nối được và không đủ 5, 6 -> False

nums = [1, 2, 3, 3, 4, 5]
cnt ban đầu = {1: 1, 2: 1, 3: 2, 4: 1, 5: 1}
bước 1: v=1 -> mở dãy 1,2,3; cnt={3: 1, 4: 1, 5: 1}; tail={3: 1}
bước 2: v=2 -> cnt[v]=0, đã dùng, bỏ qua
bước 3: v=3 -> mở dãy 3,4,5; cnt={}; tail={3: 1, 5: 1}
bước 4: v=3 -> cnt[v]=0, đã dùng, bỏ qua
bước 5: v=4 -> cnt[v]=0, đã dùng, bỏ qua
bước 6: v=5 -> cnt[v]=0, đã dùng, bỏ qua
duyệt hết -> True

nums = [1, 2, 3, 7, 8, 9, 9, 10, 11]
cnt ban đầu = {1: 1, 2: 1, 3: 1, 7: 1, 8: 1, 9: 2, 10: 1, 11: 1}
bước 1: v=1 -> mở dãy 1,2,3; cnt={7: 1, 8: 1, 9: 2, 10: 1, 11: 1}; tail={3: 1}
bước 2: v=2 -> cnt[v]=0, đã dùng, bỏ qua
bước 3: v=3 -> cnt[v]=0, đã dùng, bỏ qua
bước 4: v=7 -> mở dãy 7

Trace cách 4 (nén trạng thái) với `nums = [1,2,3,3,4,4,5,5]`: các giá trị phân biệt và số lần: `1:1, 2:1, 3:2, 4:2, 5:2`.

| `v` | `c` | `p1,p2,p3` trước | cần `p1+p2` | `rem` | `p1,p2,p3` sau |
|---|---|---|---|---|---|
| 1 | 1 | 0,0,0 | 0 | 1 | 1,0,0 |
| 2 | 1 | 1,0,0 | 1 | 0 | 0,1,0 |
| 3 | 2 | 0,1,0 | 1 | 1 | 1,0,1 |
| 4 | 2 | 1,0,1 | 1 | 1 | 0,1,1 |
| 5 | 2 | 0,1,1 | 1 | 1 | 0,0,2 |

Cuối cùng `p1 = p2 = 0` → `True`.

In [16]:
def trace_three_states(nums):
    p1 = p2 = p3 = 0
    prev = None
    print("nums =", nums)
    for v, group in groupby(nums):
        c = sum(1 for _ in group)
        if prev is not None and v != prev + 1:
            print(f"  (khoảng trống trước {v}: p1={p1}, p2={p2})")
            if p1 or p2:
                print("  -> còn dãy ngắn kẹt lại -> False")
                return False
            p1 = p2 = p3 = 0
        if c < p1 + p2:
            print(f"v={v}: c={c} < p1+p2={p1 + p2} -> False")
            return False
        rem = c - p1 - p2
        new = (max(0, rem - p3), p1, p2 + min(p3, rem))
        print(f"v={v}: c={c}, (p1,p2,p3)={(p1, p2, p3)}, rem={rem} -> {new}")
        p1, p2, p3 = new
        prev = v
    ok = p1 == 0 and p2 == 0
    print("kết thúc: p1,p2 =", (p1, p2), "->", ok)
    return ok

assert trace_three_states([1, 2, 3, 3, 4, 4, 5, 5]) is True
print()
assert trace_three_states([1, 2, 3, 4, 4, 5]) is False

nums = [1, 2, 3, 3, 4, 4, 5, 5]
v=1: c=1, (p1,p2,p3)=(0, 0, 0), rem=1 -> (1, 0, 0)
v=2: c=1, (p1,p2,p3)=(1, 0, 0), rem=0 -> (0, 1, 0)
v=3: c=2, (p1,p2,p3)=(0, 1, 0), rem=1 -> (1, 0, 1)
v=4: c=2, (p1,p2,p3)=(1, 0, 1), rem=1 -> (0, 1, 1)
v=5: c=2, (p1,p2,p3)=(0, 1, 1), rem=1 -> (0, 0, 2)
kết thúc: p1,p2 = (0, 0) -> True

nums = [1, 2, 3, 4, 4, 5]
v=1: c=1, (p1,p2,p3)=(0, 0, 0), rem=1 -> (1, 0, 0)
v=2: c=1, (p1,p2,p3)=(1, 0, 0), rem=0 -> (0, 1, 0)
v=3: c=1, (p1,p2,p3)=(0, 1, 0), rem=0 -> (0, 0, 1)
v=4: c=2, (p1,p2,p3)=(0, 0, 1), rem=2 -> (1, 0, 1)
v=5: c=1, (p1,p2,p3)=(1, 0, 1), rem=0 -> (0, 1, 0)
kết thúc: p1,p2 = (0, 1) -> False


## 9. Code nộp LeetCode

Lời giải tham lam hai bảng đếm (Cách 2):

In [17]:
from collections import Counter
from typing import List


class Solution:
    def isPossible(self, nums: List[int]) -> bool:
        cnt = Counter(nums)          # cnt[v]: còn bao nhiêu số v chưa dùng
        tail = Counter()             # tail[v]: số dãy (dài >= 3) đang kết thúc ở v
        for v in nums:
            if cnt[v] == 0:
                continue             # v đã dùng khi mở một dãy trước đó
            if tail[v - 1] > 0:      # nối vào dãy đang chờ v
                tail[v - 1] -= 1
                tail[v] += 1
                cnt[v] -= 1
            elif cnt[v + 1] > 0 and cnt[v + 2] > 0:     # mở dãy mới v, v+1, v+2
                cnt[v] -= 1
                cnt[v + 1] -= 1
                cnt[v + 2] -= 1
                tail[v + 2] += 1
            else:
                return False
        return True


sol = Solution()
for nums, expected in examples:
    got = sol.isPossible(list(nums))
    print(got)
    assert got == expected

True
True
False


## 10. Vì sao đúng

**Bổ đề 1 — nối vào luôn không tệ hơn mở dãy mới.** Xét một lời giải hợp lệ `S` và một số `v` được `S` đặt làm **phần tử đầu** của một dãy `C` (dài ≥ 3) trong khi tại thời điểm đó tồn tại dãy `D` kết thúc ở `v - 1`. Hai khả năng:

- `D` kết thúc ở `v - 1` (không có số nào nối tiếp sau). Ghép `D` và `C` thành một dãy `D + C`: vẫn liên tiếp (đuôi `v - 1`, đầu `v`), dài ≥ 3, các dãy khác giữ nguyên → hợp lệ, và `v` giờ nối vào `D`.
- `D` được một số `v'` bằng `v` nối tiếp. Hai số cùng giá trị là **không phân biệt được**, ta đổi vai: `v` nối vào `D`, `v'` làm đầu `C`. Tập các dãy (xét theo giá trị) không đổi.

Vậy luôn có lời giải mà `v` nối vào dãy đang chờ, nếu có.

**Bổ đề 2 — không nối được thì phải mở dãy `v, v+1, v+2`.** Nếu không có dãy nào kết thúc ở `v - 1` (theo các quyết định đã chọn ở các số nhỏ hơn), `v` là đầu một dãy mới và dãy đó có ít nhất `v, v+1, v+2`; ba giá trị đó phải còn chưa dùng. Ta chốt dãy này là đúng ba số đó (đuôi `v + 2`); nếu lời giải thật còn kéo dài dãy thì các số `v+3, ...` sẽ được xử lý bằng Bổ đề 1 (nối vào đuôi `v + 2` đang chờ). Nếu `v + 1` hoặc `v + 2` đã hết thì không có cách nào khác: `False`.

**Bổ đề 3 — mọi dãy trong `tail` đều dài ≥ 3.** Dãy chỉ được tạo ở bước "mở dãy" với ba phần tử, và chỉ được kéo dài thêm. Nên cuối cùng mọi dãy đều dài ≥ 3, và không cần kiểm tra độ dài ở cuối.

**Bổ đề 4 — mọi phần tử đều được dùng.** `cnt` bị trừ khi một số được dùng; vòng lặp duyệt hết `nums`. Số nào còn `cnt > 0` khi tới lượt thì hoặc nối được, hoặc mở dãy (trừ `cnt`), hoặc thất bại; số đã dùng bị bỏ qua đúng một lần cho mỗi bản sao. Không phần tử nào bị bỏ sót.

**Kết luận.** Mỗi quyết định của thuật toán đều "không mất tính khả thi" theo Bổ đề 1 và 2, nên nếu tồn tại cách chia hợp lệ thì thuật toán tìm ra (trả `True`); nếu thuật toán thất bại tại một số `v` thì mọi cách đều thất bại. Cách 3 và 4 mô phỏng cùng một ý (nối vào dãy ngắn nhất, rồi kiểm tra độ dài); tính đúng của chúng được kiểm chứng thêm bằng oracle vét cạn ở mục 12.

In [18]:
# Kiểm chứng thực nghiệm: thuật toán tham lam không bao giờ "oan" (mọi cách đều khớp oracle vét cạn)
rng = random.Random(10)
for _ in range(2000):
    n = rng.randint(1, 11)
    nums = sorted(rng.randint(-2, 4) for _ in range(n))
    assert split_two_maps(nums) == split_brute(nums), nums
print("Tham lam khớp vét cạn trên 2000 mảng ngẫu nhiên")

Tham lam khớp vét cạn trên 2000 mảng ngẫu nhiên


## 11. Time / Space Complexity

Đặt `n = len(nums)`, `m` là số giá trị phân biệt.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn DFS | mũ (chỉ cho `n` nhỏ) | số trạng thái |
| 2. Tham lam hai bảng đếm | `O(n)` | `O(m)` |
| 3. Heap độ dài | `O(n log n)` | `O(n)` |
| 4. Nén trạng thái 3 số | `O(n)` | `O(1)` |

Cách 2: mỗi phần tử duyệt đúng một lần, mỗi bước chỉ vài thao tác bảng đếm `O(1)`. Cách 3 thêm `log n` do heap. Cách 4 duyệt một lượt qua các nhóm bằng nhau và chỉ giữ ba số.

In [19]:
# Đo thực nghiệm ở kích thước tối đa n = 10^4
def make_yes(rng, n_chains, lo=-1000, hi=1000, max_len=8):
    nums = []
    for _ in range(n_chains):
        length = rng.randint(3, max_len)
        start = rng.randint(lo, hi - length + 1)
        nums.extend(range(start, start + length))
    return sorted(nums)

rng = random.Random(659)
big_yes = make_yes(rng, 2000)[:]
while len(big_yes) > 10_000:
    big_yes = make_yes(rng, 1500)
big_no = big_yes + [big_yes[-1]]            # thêm một số trùng: có thể phá vỡ tính khả thi
big_cases = {
    "yes ngẫu nhiên": big_yes,
    "yes lặp 1..3333 x3": sorted(list(range(1, 3334)) * 3),
    "thêm một số trùng": sorted(big_no),
    "toàn số bằng nhau": [7] * 10_000,
}
for name, nums in big_cases.items():
    assert len(nums) <= 10_000
    t0 = time.perf_counter()
    r2 = split_two_maps(nums)
    r3 = split_heap(nums)
    r4 = split_three_states(nums)
    dt = time.perf_counter() - t0
    assert r2 == r3 == r4 and dt < 5, (name, r2, r3, r4)
    print(f"{name:22s} n={len(nums):5d} -> {r2}, ba cách khớp nhau")

yes ngẫu nhiên         n= 8291 -> True, ba cách khớp nhau


yes lặp 1..3333 x3     n= 9999 -> True, ba cách khớp nhau


thêm một số trùng      n= 8292 -> False, ba cách khớp nhau
toàn số bằng nhau      n=10000 -> False, ba cách khớp nhau


## 12. Bộ test

Chia 4 nhóm: (a) ví dụ đề và edge case có đáp án tính tay; (b) phản ví dụ cho các heuristic sai; (c) đối chiếu mọi cách với **oracle vét cạn** trên mọi mảng nhỏ; (d) test ngẫu nhiên có seed (cả mảng sinh sẵn từ các dãy hợp lệ → đáp án chắc chắn `True`).

### 12.1. Test cố định

In [20]:
solvers = {
    "brute": split_brute,
    "two_maps": split_two_maps,
    "heap": split_heap,
    "three_states": split_three_states,
    "submit": lambda nums: Solution().isPossible(list(nums)),
}

fixed_cases = [
    ([1, 2, 3, 3, 4, 5], True),               # ví dụ 1
    ([1, 2, 3, 3, 4, 4, 5, 5], True),         # ví dụ 2
    ([1, 2, 3, 4, 4, 5], False),              # ví dụ 3
    ([1], False),                             # một phần tử
    ([1, 2], False),                          # quá ngắn
    ([1, 2, 3], True),                        # đúng một dãy tối thiểu
    ([1, 1, 1], False),                       # toàn số bằng nhau
    ([1, 2, 2, 3], False),                    # 3 giá trị liên tiếp nhưng có trùng: không đủ
    ([1, 2, 3, 4, 5, 6], True),               # một dãy dài (hoặc hai dãy 3)
    ([1, 2, 3, 5, 6, 7], True),               # hai cụm tách rời, mỗi cụm 3
    ([1, 2, 3, 5, 6], False),                 # cụm sau chỉ dài 2
    ([-3, -2, -1], True),                     # số âm
    ([1, 2, 3, 3, 4, 4, 5], True),            # 1,2,3,4 và 3,4,5
    ([1, 2, 3, 4, 5, 5, 6, 7], True),         # 1..5 và 5,6,7
    ([1, 1, 2, 2, 3, 3], True),               # hai dãy 1,2,3
    ([1, 1, 2, 2, 3, 3, 4, 4, 5, 5], True),   # hai dãy 1..5 hoặc ...
]
for nums, expected in fixed_cases:
    for name, f in solvers.items():
        got = f(list(nums))
        assert got == expected, (name, nums, got, expected)
print("Test cố định: OK (", len(fixed_cases), "bộ x", len(solvers), "lời giải )")

Test cố định: OK ( 16 bộ x 5 lời giải )


### 12.2. Phản ví dụ cho các heuristic sai

- *"Chỉ cần mỗi cụm giá trị phân biệt liên tiếp dài ≥ 3"* (bỏ qua số trùng): `[1,1,2,3]` có cụm `1,2,3` dài 3 nhưng thừa một số `1` không thể xếp vào dãy nào → thực ra `False`.
- *"Nối vào dãy **dài nhất** đang chờ"* (thay vì ngắn nhất): `[1,2,2,3,3,4]`: có thể chia `1,2,3` và `2,3,4` (`True`), nhưng nối vào dãy dài sẽ làm dãy ngắn bị bỏ rơi → sai.

In [21]:
def wrong_runs_only(nums):
    values = sorted(set(nums))
    run = 1
    for a, b in zip(values, values[1:]):
        if b == a + 1:
            run += 1
        else:
            if run < 3:
                return False
            run = 1
    return run >= 3


def wrong_longest_first(nums):
    d = defaultdict(list)
    for v in nums:
        h = d[v - 1]
        if h:
            h.sort()
            d[v].append(h.pop() + 1)         # SAI: nối vào dãy DÀI nhất
        else:
            d[v].append(1)
    return all(not h or min(h) >= 3 for h in d.values())


c1 = [1, 1, 2, 3]
print("runs_only:", wrong_runs_only(c1), "| đúng:", split_brute(c1))
assert wrong_runs_only(c1) is True and split_brute(c1) is False and split_two_maps(c1) is False

c2 = [1, 2, 2, 3, 3, 4]
print("longest_first:", wrong_longest_first(c2), "| đúng:", split_brute(c2))
assert wrong_longest_first(c2) is False and split_brute(c2) is True and Solution().isPossible(c2) is True

runs_only: True | đúng: False
longest_first: False | đúng: True


### 12.3. Đối chiếu với oracle trên **mọi** mảng nhỏ

Vét mọi mảng không giảm với độ dài ≤ 9 và giá trị trong `1..5`. Với mỗi mảng, mọi cách đều phải khớp `split_brute`.

In [22]:
checked = 0
yes_count = 0
for n in range(1, 10):
    for nums in combinations_with_replacement(range(1, 6), n):
        nums = list(nums)
        expected = split_brute(nums)
        for name, f in solvers.items():
            assert f(nums) == expected, (name, nums, f(nums), expected)
        checked += 1
        yes_count += expected
print("Vét hết mảng nhỏ:", checked, "mảng, trong đó", yes_count, "mảng chia được; mọi cách khớp oracle")

Vét hết mảng nhỏ: 2001 mảng, trong đó 35 mảng chia được; mọi cách khớp oracle


### 12.4. Test ngẫu nhiên có seed

Nhóm 1: mảng ngẫu nhiên nhỏ (có oracle). Nhóm 2: mảng **sinh từ các dãy hợp lệ** (đáp án chắc chắn `True`), cỡ lớn. Nhóm 3: mảng lớn ngẫu nhiên, so các cách nhanh với nhau.

In [23]:
rng = random.Random(659)
for _ in range(2000):
    n = rng.randint(1, 12)
    nums = sorted(rng.randint(-2, 5) for _ in range(n))
    expected = split_brute(nums)
    for name, f in solvers.items():
        assert f(nums) == expected, (name, nums)
print("Ngẫu nhiên cỡ nhỏ (seed 659): 2000 mảng, khớp oracle")

Ngẫu nhiên cỡ nhỏ (seed 659): 2000 mảng, khớp oracle


In [24]:
rng = random.Random(2026)
for _ in range(300):
    nums = make_yes(rng, rng.randint(1, 40), max_len=rng.choice([3, 4, 8]))
    assert 1 <= len(nums) <= 10_000
    for name, f in solvers.items():
        if name == "brute":
            continue                         # vét cạn không chạy nổi mảng lớn
        assert f(nums) is True, (name, nums)
print("Mảng sinh từ dãy hợp lệ: 300 mảng, mọi cách trả True")

Mảng sinh từ dãy hợp lệ: 300 mảng, mọi cách trả True


In [25]:
rng = random.Random(77)
fast = {k: v for k, v in solvers.items() if k != "brute"}
yes_seen = no_seen = 0
for _ in range(500):
    n = rng.randint(1, 3000)
    span = rng.choice([3, 10, 50, 1000])
    nums = sorted(rng.randint(-span, span) for _ in range(n))
    results = {name: f(nums) for name, f in fast.items()}
    assert len(set(results.values())) == 1, (results, n, span)
    yes_seen += results["two_maps"]
    no_seen += not results["two_maps"]
print("Ngẫu nhiên cỡ lớn: 500 mảng khớp nhau; True:", yes_seen, "| False:", no_seen)

Ngẫu nhiên cỡ lớn: 500 mảng khớp nhau; True: 9 | False: 491


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ nhìn các giá trị phân biệt, bỏ qua số trùng

`[1,1,2,3]` có cụm giá trị `1,2,3` nhưng còn dư một số `1` không thuộc dãy nào. Đề yêu cầu **mọi phần tử** phải được dùng. Phải đếm số lần xuất hiện.

In [26]:
print("sai:", wrong_runs_only([1, 1, 2, 3]), "| đúng:", Solution().isPossible([1, 1, 2, 3]))
assert Solution().isPossible([1, 1, 2, 3]) is False

sai: True | đúng: False


### Lỗi 2 — Nối vào dãy **dài nhất** thay vì ngắn nhất (khi dùng heap/độ dài)

Dãy ngắn (độ dài 1, 2) là dãy chưa đạt yêu cầu, cần được nối trước. Chọn dãy dài nhất để nối làm dãy ngắn bị bỏ rơi. Đã thấy ở mục 12.2: `[1,2,2,3,3,4]` cho kết quả sai.

In [27]:
print("sai (dài nhất):", wrong_longest_first([1, 2, 2, 3, 3, 4]), "| đúng (ngắn nhất):", split_heap([1, 2, 2, 3, 3, 4]))
assert wrong_longest_first([1, 2, 2, 3, 3, 4]) is False and split_heap([1, 2, 2, 3, 3, 4]) is True

sai (dài nhất): False | đúng (ngắn nhất): True


### Lỗi 3 — Quên kiểm tra độ dài ≥ 3 ở cuối (cách heap)

Nếu bỏ bước `all(not h or h[0] >= 3 ...)` mà chỉ trả `True` sau khi duyệt, các dãy dài 1 hoặc 2 vẫn được chấp nhận. Ví dụ `[1,2]` hay ví dụ 3 `[1,2,3,4,4,5]` đều bị trả `True`.

In [28]:
def wrong_no_final_check(nums):
    d = defaultdict(list)
    for v in nums:
        h = d[v - 1]
        if h:
            heapq.heappush(d[v], heapq.heappop(h) + 1)
        else:
            heapq.heappush(d[v], 1)
    return True                            # SAI: quên kiểm tra mọi dãy dài >= 3

print("sai:", wrong_no_final_check([1, 2]), wrong_no_final_check([1, 2, 3, 4, 4, 5]),
      "| đúng:", split_heap([1, 2]), split_heap([1, 2, 3, 4, 4, 5]))
assert wrong_no_final_check([1, 2]) is True and split_heap([1, 2]) is False

sai: True True | đúng: False False


### Lỗi 4 — Quên bỏ qua số đã được dùng (`cnt[v] == 0`)

Khi mở dãy `v, v+1, v+2` ta **trừ** `cnt[v+1]` và `cnt[v+2]`, nhưng vòng lặp vẫn sẽ gặp các số `v+1`, `v+2` trong `nums`. Nếu không kiểm tra `cnt[v] == 0` để bỏ qua, ta lại xử lý chúng như số mới và thường thất bại.

In [29]:
def wrong_no_skip(nums):
    cnt = Counter(nums)
    tail = Counter()
    for v in nums:
        # SAI: thiếu "if cnt[v] == 0: continue"
        if tail[v - 1] > 0:
            tail[v - 1] -= 1
            tail[v] += 1
            cnt[v] -= 1
        elif cnt[v + 1] > 0 and cnt[v + 2] > 0:
            cnt[v] -= 1
            cnt[v + 1] -= 1
            cnt[v + 2] -= 1
            tail[v + 2] += 1
        else:
            return False
    return True

print("sai:", wrong_no_skip([1, 2, 3]), "| đúng:", Solution().isPossible([1, 2, 3]))
assert wrong_no_skip([1, 2, 3]) is False and Solution().isPossible([1, 2, 3]) is True

sai: False | đúng: True


### Lỗi 5 — Tham lam "mở dãy mới trước" mà không kiểm tra đủ `v+1`, `v+2`

Mở dãy mới khi **chưa** chắc có `v + 1`, `v + 2` làm ta tạo ra dãy ngắn mà không biết. Cách 2 luôn kiểm tra `cnt[v + 1] > 0 and cnt[v + 2] > 0` trước khi mở; nếu không đủ thì trả `False` ngay vì không còn lựa chọn nào khác (Bổ đề 2).

## 14. Mẹo & nhận dạng pattern

- **"Chia mảng thành các nhóm liên tiếp có độ dài tối thiểu"** → tham lam: **nối vào nhóm đang chờ trước, chỉ mở nhóm mới khi buộc phải**; đếm "đuôi" bằng bảng băm.
- Dùng **hai bảng đếm**: `cnt` (số còn lại) và `tail` (đuôi các nhóm). Không cần biết độ dài nhóm khi mọi nhóm sinh ra đã đạt độ dài tối thiểu.
- Nếu dùng cách "mở nhóm độ dài 1 rồi sửa sau" thì phải **nối vào nhóm ngắn nhất** (heap) và kiểm tra độ dài ở cuối.
- **Số trùng** quyết định bài này: đừng chỉ nhìn các giá trị phân biệt.
- Bài cùng họ: Hand of Straights (846), Divide Array in Sets of K Consecutive Numbers (1296), Longest Consecutive Sequence (128), Task Scheduler (621), Reorganize String (767).
- Khi mảng **chưa** sắp xếp: sắp xếp trước (hoặc dùng `Counter` rồi duyệt khóa theo thứ tự tăng).
- Luôn có oracle vét cạn cho `n` nhỏ để kiểm chứng tham lam.

## 15. Biến thể

### 15.1. Trả về cách chia cụ thể

Nếu đề yêu cầu in ra các dãy, ta lưu các dãy thay vì chỉ đếm đuôi: `tails[v]` là danh sách các dãy (list) đang kết thúc ở `v`.

In [30]:
def split_construct(nums):
    cnt = Counter(nums)
    tails = defaultdict(list)           # tails[v]: các dãy đang kết thúc ở v
    chains = []
    for v in nums:
        if cnt[v] == 0:
            continue
        if tails[v - 1]:
            chain = tails[v - 1].pop()
            chain.append(v)
            tails[v].append(chain)
            cnt[v] -= 1
        elif cnt[v + 1] > 0 and cnt[v + 2] > 0:
            cnt[v] -= 1
            cnt[v + 1] -= 1
            cnt[v + 2] -= 1
            chain = [v, v + 1, v + 2]
            chains.append(chain)
            tails[v + 2].append(chain)
        else:
            return None
    return chains


def valid_split(nums, chains):
    flat = sorted(x for c in chains for x in c)
    return flat == sorted(nums) and all(len(c) >= 3 and all(b == a + 1 for a, b in zip(c, c[1:])) for c in chains)


print(split_construct([1, 2, 3, 3, 4, 4, 5, 5]))
print(split_construct([1, 2, 3, 4, 4, 5]))
rng = random.Random(15)
for _ in range(500):
    nums = sorted(rng.randint(-2, 5) for _ in range(rng.randint(1, 12)))
    res = split_construct(nums)
    assert (res is not None) == split_brute(nums)
    if res is not None:
        assert valid_split(nums, res), (nums, res)
print("OK: cách chia dựng ra luôn hợp lệ")

[[1, 2, 3, 4, 5], [3, 4, 5]]
None
OK: cách chia dựng ra luôn hợp lệ


### 15.2. Độ dài tối thiểu `m` bất kỳ (thay vì 3)

Đổi `3` thành `m`: khi không nối được, ta cần `cnt[v + i] > 0` với mọi `i = 1..m-1` để mở dãy `m` phần tử. Đây là bài tập 5 ở mục 17.

### 15.3. Mọi dãy phải dài **đúng** `k` (Hand of Straights / Divide Array in Sets of K Consecutive Numbers)

Khi độ dài cố định `k`, ta lấy số nhỏ nhất còn lại `v` và **bắt buộc** nó mở dãy `v, v+1, ..., v+k-1`. Độ phức tạp `O(n)` với bảng đếm (duyệt theo thứ tự tăng dần).

In [31]:
def split_exact_k(nums, k):
    cnt = Counter(nums)
    for v in nums:                        # nums đã sắp xếp: v là số nhỏ nhất còn lại khi cnt[v] > 0
        c = cnt[v]
        if c == 0:
            continue
        for i in range(k):
            if cnt[v + i] < c:
                return False
            cnt[v + i] -= c               # mở c dãy cùng lúc bắt đầu từ v
    return True

print(split_exact_k([1, 2, 3, 3, 4, 5], 3), split_exact_k([1, 2, 3, 4, 5], 3), split_exact_k([1, 1, 2, 2, 3, 3], 3))
rng = random.Random(153)
for _ in range(1500):
    k = rng.randint(1, 4)
    nums = sorted(rng.randint(0, 5) for _ in range(rng.randint(1, 10)))
    assert split_exact_k(nums, k) == split_brute(nums, minlen=k, maxlen=k), (nums, k)
print("OK: khớp vét cạn với độ dài đúng k")

True False True
OK: khớp vét cạn với độ dài đúng k


### 15.4. Các biến thể khác (chỉ mô tả)

- **Mảng chưa sắp xếp**: sắp xếp trước, hoặc duyệt các giá trị phân biệt theo thứ tự tăng từ `Counter`.
- **Tối thiểu số dãy / tối đa số dãy**: heap độ dài với quy tắc nối khác nhau (ví dụ nối vào dãy **dài nhất** để giảm tổng số dãy chưa chắc đúng; cần chứng minh riêng).
- **Dãy có thể "nhảy" cách `d`** (hơn kém `d` thay vì 1): chia theo lớp đồng dư `mod d` rồi giải từng lớp như bài gốc.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn DFS | thử mọi cách gán số vào dãy | mũ | số trạng thái | chỉ làm oracle với `n` nhỏ |
| 2. Hai bảng đếm | nối vào dãy chờ, không thì mở `v,v+1,v+2` | `O(n)` | `O(m)` | **cách nên nộp**: ngắn, nhanh, chứng minh gọn |
| 3. Heap độ dài | nối vào dãy **ngắn nhất**, kiểm tra độ dài cuối | `O(n log n)` | `O(n)` | khi không muốn "nhìn trước"; mở rộng sang biến thể có độ dài |
| 4. Nén trạng thái | giữ số dãy độ dài 1, 2, ≥3 theo từng giá trị | `O(n)` | `O(1)` | khi cần bộ nhớ tối thiểu; khó nghĩ hơn |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print(Counter([1, 2, 2, 3, 3, 3])[3], Counter([1, 2])[9])
print([(v, len(list(g))) for v, g in groupby([4, 4, 5, 7, 7, 7])])
print(all(l >= 3 for l in [3, 4, 2]), any(l < 3 for l in [3, 4, 2]))
```

In [32]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `nums = [1,2,3,4,5,5,6,7]`, chạy tay cách 2 (hai bảng đếm): ở mỗi `v` ghi hành động (nối / mở dãy / bỏ qua). Cho biết đáp án và các đuôi còn tồn tại (`tail` khác 0) khi kết thúc.

In [33]:
# Bài 2: điền kết quả tính tay
hand_answer = None            # True / False
hand_final_tail = None        # dict các đuôi khác 0, ví dụ {5: 1}

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm theo khuôn mẫu hai bảng đếm.

In [34]:
def exercise_3_is_possible(nums: List[int]) -> bool:
    cnt = Counter(nums)
    tail = Counter()
    for v in nums:
        # TODO 1: nếu cnt[v] == 0 thì bỏ qua (continue)
        # TODO 2: nếu tail[v - 1] > 0: nối v vào dãy đó (cập nhật tail[v-1], tail[v], cnt[v])
        # TODO 3: ngược lại nếu cnt[v+1] > 0 và cnt[v+2] > 0: mở dãy v, v+1, v+2 (trừ cnt, tail[v+2] += 1)
        # TODO 4: ngược lại trả False
        pass
    return None


# assert exercise_3_is_possible([1, 2, 3, 3, 4, 5]) is True
# assert exercise_3_is_possible([1, 2, 3, 4, 4, 5]) is False
# assert exercise_3_is_possible([1, 2, 3, 3, 4, 4, 5, 5]) is True

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại cách 4, hãy tự viết `exercise_4_three_states(nums)`: giữ ba số `p1, p2, p3` (số dãy độ dài 1, 2, ≥ 3 kết thúc ở giá trị liền trước) khi duyệt các giá trị phân biệt tăng dần.

In [35]:
def exercise_4_three_states(nums: List[int]) -> bool:
    # TODO
    pass


# assert exercise_4_three_states([1, 2, 3, 3, 4, 5]) is True
# assert exercise_4_three_states([1, 2, 3, 4, 4, 5]) is False
# assert exercise_4_three_states([1, 2, 3, 7, 8, 9]) is True

### Bài 5 — Mini challenge: độ dài tối thiểu `m` tổng quát (khó)

Viết `exercise_5_min_length(nums, m)`: mọi dãy con liên tiếp phải dài **ít nhất `m`** (`m >= 1`). Với `m = 3` trùng bài gốc. Dùng tham lam hai bảng đếm tổng quát hóa.

```python
exercise_5_min_length([1, 2, 3, 3, 4, 5], 3) -> True
exercise_5_min_length([1, 2, 3, 4], 5) -> False
exercise_5_min_length([1, 2, 3, 4, 5], 5) -> True
exercise_5_min_length([1, 1, 2, 2], 2) -> True
```

Gợi ý: khi không nối được, mở dãy cần `cnt[v + i] > 0` với mọi `i` từ `1` đến `m - 1`, rồi tăng `tail[v + m - 1]`.

In [36]:
def exercise_5_min_length(nums: List[int], m: int) -> bool:
    # TODO
    pass


# assert exercise_5_min_length([1, 2, 3, 3, 4, 5], 3) is True
# assert exercise_5_min_length([1, 2, 3, 4], 5) is False
# assert exercise_5_min_length([1, 2, 3, 4, 5], 5) is True
# assert exercise_5_min_length([1, 1, 2, 2], 2) is True

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `Counter` đếm số lần; khóa vắng mặt cho `0`; `groupby` nhóm các số bằng nhau liền kề.
- Mức 2: `3` xuất hiện 3 lần; `[4,4,5,7,7,7]` có ba nhóm.
- Mức 3: dòng 1 `3 0`; dòng 2 `[(4, 2), (5, 1), (7, 3)]`; dòng 3 `False True`.

**Bài 2**
- Mức 1: `1` mở dãy `1,2,3`; `2`, `3` bị bỏ qua.
- Mức 2: `4` nối vào đuôi 3, `5` (lần 1) nối vào đuôi 4, `5` (lần 2) không có đuôi 4 nữa nên phải mở dãy `5,6,7`.
- Mức 3: đáp án `True`; `tail` cuối `{5: 1, 7: 1}`.

**Bài 3**
- Mức 1: giống code mục 9, từng nhánh `if / elif / else`.
- Mức 2: nối: `tail[v-1] -= 1; tail[v] += 1; cnt[v] -= 1`.
- Mức 3: mở dãy: trừ `cnt[v]`, `cnt[v+1]`, `cnt[v+2]` và `tail[v+2] += 1`; duyệt xong `return True`.

**Bài 4**
- Mức 1: với mỗi giá trị phân biệt `v` có `c` lần, cần `c >= p1 + p2`.
- Mức 2: `rem = c - p1 - p2`; `p1, p2, p3 = max(0, rem - p3), p1, p2 + min(p3, rem)`.
- Mức 3: khi `v` không liền sau giá trị trước, yêu cầu `p1 == p2 == 0` rồi đặt lại ba số về `0`; cuối trả `p1 == 0 and p2 == 0`.

**Bài 5**
- Mức 1: thay hằng `3` bằng `m` ở nhánh mở dãy.
- Mức 2: kiểm tra `all(cnt[v + i] > 0 for i in range(1, m))`.
- Mức 3: mở dãy: `for i in range(m): cnt[v + i] -= 1` rồi `tail[v + m - 1] += 1`; ca `m = 1` thì mọi phần tử tự thành dãy (hàm trả `True`).

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [37]:
print(Counter([1, 2, 2, 3, 3, 3])[3], Counter([1, 2])[9])
print([(v, len(list(g))) for v, g in groupby([4, 4, 5, 7, 7, 7])])
print(all(l >= 3 for l in [3, 4, 2]), any(l < 3 for l in [3, 4, 2]))

assert (Counter([1, 2, 2, 3, 3, 3])[3], Counter([1, 2])[9]) == (3, 0)
assert [(v, len(list(g))) for v, g in groupby([4, 4, 5, 7, 7, 7])] == [(4, 2), (5, 1), (7, 3)]
assert (all(l >= 3 for l in [3, 4, 2]), any(l < 3 for l in [3, 4, 2])) == (False, True)
print("Bài 1 OK")

3 0
[(4, 2), (5, 1), (7, 3)]
False True
Bài 1 OK


### Đáp án Bài 2

`1`: mở `1,2,3` (tail 3); `2`, `3`: bỏ qua; `4`: nối (tail 4); `5` (lần 1): nối (tail 5); `5` (lần 2): `tail[4] = 0`, có `6`, `7` → mở `5,6,7` (tail 7). Kết thúc `True`, `tail = {5: 1, 7: 1}`.

In [38]:
def run_two_maps_with_tail(nums):
    cnt = Counter(nums)
    tail = Counter()
    for v in nums:
        if cnt[v] == 0:
            continue
        if tail[v - 1] > 0:
            tail[v - 1] -= 1
            tail[v] += 1
            cnt[v] -= 1
        elif cnt[v + 1] > 0 and cnt[v + 2] > 0:
            cnt[v] -= 1
            cnt[v + 1] -= 1
            cnt[v + 2] -= 1
            tail[v + 2] += 1
        else:
            return False, {}
    return True, {k: c for k, c in sorted(tail.items()) if c}

ans2, tail2 = run_two_maps_with_tail([1, 2, 3, 4, 5, 5, 6, 7])
print(ans2, tail2)
assert ans2 is True and tail2 == {5: 1, 7: 1}
assert split_brute([1, 2, 3, 4, 5, 5, 6, 7]) is True
print("Bài 2 OK")

True {5: 1, 7: 1}
Bài 2 OK


### Đáp án Bài 3

In [39]:
def exercise_3_solution(nums: List[int]) -> bool:
    cnt = Counter(nums)
    tail = Counter()
    for v in nums:
        if cnt[v] == 0:
            continue
        if tail[v - 1] > 0:
            tail[v - 1] -= 1
            tail[v] += 1
            cnt[v] -= 1
        elif cnt[v + 1] > 0 and cnt[v + 2] > 0:
            cnt[v] -= 1
            cnt[v + 1] -= 1
            cnt[v + 2] -= 1
            tail[v + 2] += 1
        else:
            return False
    return True


assert exercise_3_solution([1, 2, 3, 3, 4, 5]) is True
assert exercise_3_solution([1, 2, 3, 4, 4, 5]) is False
assert exercise_3_solution([1, 2, 3, 3, 4, 4, 5, 5]) is True
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [40]:
def exercise_4_solution(nums: List[int]) -> bool:
    p1 = p2 = p3 = 0
    prev = None
    for v, group in groupby(nums):
        c = len(list(group))
        if prev is not None and v != prev + 1:
            if p1 or p2:
                return False
            p1 = p2 = p3 = 0
        if c < p1 + p2:
            return False
        rem = c - p1 - p2
        p1, p2, p3 = max(0, rem - p3), p1, p2 + min(p3, rem)
        prev = v
    return p1 == 0 and p2 == 0


assert exercise_4_solution([1, 2, 3, 3, 4, 5]) is True
assert exercise_4_solution([1, 2, 3, 4, 4, 5]) is False
assert exercise_4_solution([1, 2, 3, 7, 8, 9]) is True
rng = random.Random(4)
for _ in range(1000):
    nums = sorted(rng.randint(-2, 5) for _ in range(rng.randint(1, 11)))
    assert exercise_4_solution(nums) == split_brute(nums), nums
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [41]:
def exercise_5_solution(nums: List[int], m: int) -> bool:
    cnt = Counter(nums)
    tail = Counter()
    for v in nums:
        if cnt[v] == 0:
            continue
        if tail[v - 1] > 0:
            tail[v - 1] -= 1
            tail[v] += 1
            cnt[v] -= 1
        elif all(cnt[v + i] > 0 for i in range(1, m)):
            for i in range(m):
                cnt[v + i] -= 1
            tail[v + m - 1] += 1
        else:
            return False
    return True


assert exercise_5_solution([1, 2, 3, 3, 4, 5], 3) is True
assert exercise_5_solution([1, 2, 3, 4], 5) is False
assert exercise_5_solution([1, 2, 3, 4, 5], 5) is True
assert exercise_5_solution([1, 1, 2, 2], 2) is True
rng = random.Random(5)
for _ in range(2000):
    m = rng.randint(1, 5)
    nums = sorted(rng.randint(0, 5) for _ in range(rng.randint(1, 11)))
    assert exercise_5_solution(nums, m) == split_brute(nums, minlen=m), (nums, m)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def isPossible(self, nums: List[int]) -> bool:
        cnt = Counter(nums)          # số còn lại của mỗi giá trị
        tail = Counter()             # số dãy (dài >= 3) đang kết thúc ở mỗi giá trị
        for v in nums:
            if cnt[v] == 0:
                continue             # đã dùng
            if tail[v - 1] > 0:      # ưu tiên nối vào dãy đang chờ
                tail[v - 1] -= 1
                tail[v] += 1
                cnt[v] -= 1
            elif cnt[v + 1] > 0 and cnt[v + 2] > 0:   # không thì mở dãy mới v, v+1, v+2
                cnt[v] -= 1
                cnt[v + 1] -= 1
                cnt[v + 2] -= 1
                tail[v + 2] += 1
            else:
                return False
        return True
```

Câu thần chú: **"có dãy đang chờ `v` thì nối; không thì mở `v, v+1, v+2`; không mở được thì `False`."**

### Checklist trước khi submit

- [ ] Có `if cnt[v] == 0: continue` ở đầu vòng lặp.
- [ ] Ưu tiên **nối vào dãy đang chờ** (`tail[v-1]`) **trước** khi mở dãy mới.
- [ ] Mở dãy mới phải kiểm tra **cả** `cnt[v+1]` và `cnt[v+2]`, rồi trừ cả ba và tăng `tail[v+2]`.
- [ ] Không bỏ qua số trùng: dùng `Counter`, không chỉ nhìn tập giá trị phân biệt.
- [ ] Nếu dùng heap độ dài: nối vào dãy **ngắn nhất** và kiểm tra mọi dãy dài ≥ 3 ở cuối.
- [ ] Test: ví dụ đề, `[1]`, `[1,2]`, `[1,1,1]`, `[1,1,2,3]`, hai cụm tách rời, số âm.
- [ ] Skeleton đúng: `class Solution`, method `isPossible(self, nums: List[int]) -> bool`.